# 提示词评估系统 - 代码调用顺序

## 📋 整体流程概览

本 notebook 实现了一个完整的提示词评估系统，用于测试和评估 AI 提示词的质量。以下是代码的执行顺序和调用关系：

## 🔄 执行流程

### 阶段 1：环境准备
1. **导入库** (Cell 1)
   - 导入所有必要的 Python 库

2. **初始化客户端和辅助函数** (Cell 3)
   - 加载环境变量
   - 初始化 Anthropic API 客户端
   - 定义 `add_user_message()`, `add_assistant_message()`, `chat()` 辅助函数

3. **定义报告生成器** (Cell 5)
   - 定义 `generate_prompt_evaluation_report()` 函数，用于生成 HTML 报告

### 阶段 2：核心类实现
4. **实现 PromptEvaluator 类** (Cell 7)
   - 定义核心评估类及其所有方法

### 阶段 3：使用评估系统
5. **创建评估器实例** (Cell 9)
   ```python
   evaluator = PromptEvaluator(max_concurrent_tasks=1)
   ```

6. **生成测试数据集** (Cell 11)
   ```python
   dataset = evaluator.generate_dataset(...)
   ```
   - 内部调用：`generate_unique_ideas()` → `generate_test_case()` (并发)
   - 输出：`dataset.json` 文件

7. **定义要评估的提示词函数** (Cell 13)
   ```python
   def run_prompt(prompt_inputs):
       # 构建提示词并调用 chat()
   ```

8. **运行完整评估** (Cell 15)
   ```python
   results = evaluator.run_evaluation(...)
   ```
   - 内部调用流程：
     1. 读取 `dataset.json`
     2. 对每个测试用例并发执行：
        - `run_test_case()` → `run_prompt()` → `grade_output()`
     3. 生成 `output.json` 和 `output.html`

## 🔗 函数调用关系图

```
evaluator.generate_dataset()
    ├── generate_unique_ideas()  # 生成测试想法
    └── generate_test_case()     # 为每个想法生成测试用例（并发）
        └── chat()               # 调用 API 生成测试用例

evaluator.run_evaluation()
    └── run_test_case()          # 对每个测试用例（并发）
        ├── run_prompt()         # 执行提示词，获取输出
        │   └── chat()           # 调用 API 生成输出
        └── grade_output()       # 评估输出质量
            └── chat()           # 调用 API 进行评分

generate_prompt_evaluation_report()
    └── 生成 HTML 报告
```

## ⚙️ 关键参数说明

- **`max_concurrent_tasks`**: 控制并发数量，避免 API 速率限制
- **`num_cases`**: 测试用例数量，建议从少量开始测试
- **`extra_criteria`**: 强制性评估标准，违反将导致低分（≤3）

# 导入必要的库

本 cell 导入提示词评估系统所需的所有 Python 库：
- `json`: 用于处理 JSON 数据格式
- `concurrent.futures`: 用于并发执行任务，提高评估效率
- `re`: 用于正则表达式处理
- `textwrap.dedent`: 用于去除多行字符串的缩进
- `statistics.mean`: 用于计算平均分数
- `dotenv`: 用于加载环境变量
- `anthropic`: Anthropic API 客户端库

In [109]:
# Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

# 客户端初始化和辅助函数

本 cell 完成以下任务：

1. **加载环境变量**：从 `.env` 文件加载配置
2. **初始化 Anthropic 客户端**：配置 API 密钥和基础 URL（支持本地代理或远程代理）
3. **定义辅助函数**：
   - `add_user_message()`: 向消息列表添加用户消息
   - `add_assistant_message()`: 向消息列表添加助手消息
   - `chat()`: 核心聊天函数，调用 API 并处理响应，支持系统提示、温度控制和停止序列

In [110]:
# Client Initialization and helper functions

load_dotenv()
# model = "claude-haiku-4-5"
base_url = "http://127.0.0.1:8045"
api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
client = Anthropic(api_key=api_key, base_url=base_url)
model = "gemini-3-flash"
# model = "claude-opus-4-5-thinking"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], max_tokens=4000):
    """
    聊天函数，调用 API 获取响应
    
    参数:
    - messages: 消息列表
    - system: 系统提示（可选）
    - temperature: 温度参数，控制随机性（0.0-1.0）
    - stop_sequences: 停止序列列表
    - max_tokens: 最大输出 token 数（默认 4000，可根据需要调整）
    """
    params = {
        "model": model,
        "max_tokens": max_tokens,  # 增加到 4000，支持更长的输出
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    text = message.content[0].text
    print(f"""chat text: {text}""")
    # 客户端处理停止序列（代理可能不支持）
    if stop_sequences:
        for stop_seq in stop_sequences:
            if stop_seq in text:
                text = text.split(stop_seq)[0]
                break

    return text

# 报告生成器

`generate_prompt_evaluation_report()` 函数用于生成 HTML 格式的评估报告。

**功能**：
- 计算评估统计信息：总测试用例数、平均分数、通过率（分数 ≥ 7）
- 生成美观的 HTML 报告，包含：
  - 统计摘要（总测试用例、平均分数、通过率）
  - 详细的测试结果表格（场景、提示输入、解决方案标准、输出、分数、推理）
  - 根据分数自动着色（高分绿色、中分黄色、低分红色）

In [111]:
# Report Builder
def generate_prompt_evaluation_report(evaluation_results, extra_criteria=None):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    # 生成额外标准的 HTML 部分（用于表格中的 Solution Criteria 列）
    extra_criteria_html = ""
    if extra_criteria:
        extra_criteria_html = f"""<div class="extra-criteria-section"><strong>Extra Criteria (强制性要求):</strong><br>{extra_criteria.strip().replace(chr(10), '<br>')}</div>"""

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            .extra-criteria-section {{
                margin-top: 12px;
                padding: 10px;
                background-color: #fff3e0;
                border-left: 3px solid #ff9800;
                border-radius: 3px;
                font-size: 13px;
                line-height: 1.6;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}
            
            .prompt-cell pre {{
                background-color: #e3f2fd;
                border: 1px solid #90caf9;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 13px;
                line-height: 1.4;
                color: #1565c0;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        # 优先使用完整的 prompt，如果没有则回退到 prompt_inputs
        if result.get("prompt"):
            prompt_html = f'<pre>{result["prompt"]}</pre>'
        else:
            prompt_html = "<br>".join(
                [
                    f"<strong>{key}:</strong> {value}"
                    for key, value in result["test_case"]["prompt_inputs"].items()
                ]
            )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-cell">{prompt_html}</td>
                <td class="criteria">• {criteria_string}{extra_criteria_html}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

# PromptEvaluator 类实现

这是提示词评估系统的核心类，包含以下主要方法：

## 主要功能

1. **`generate_unique_ideas()`**: 生成独特的测试想法
   - 基于任务描述生成多样化的测试场景想法
   - 返回 JSON 数组格式的想法列表

2. **`generate_test_case()`**: 生成单个测试用例
   - 根据任务描述和特定想法生成详细的测试用例
   - 包含提示输入和解决方案评估标准

3. **`generate_dataset()`**: 生成测试数据集
   - 批量生成多个测试用例
   - 使用并发处理提高效率
   - 保存为 JSON 文件

4. **`grade_output()`**: 对输出进行评分
   - 使用 AI 模型评估测试用例的输出
   - 返回包含优势、劣势、推理和分数的结构化结果

5. **`run_test_case()`**: 运行单个测试用例
   - 执行提示词并获取输出
   - 对输出进行评分

6. **`run_evaluation()`**: 运行完整评估
   - 对数据集中的所有测试用例进行评估
   - 生成 JSON 和 HTML 报告

In [112]:
# PromptEvaluator Implementation
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}", str(variables[placeholder])
                )

        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """基于任务描述生成测试用例的独特想法列表"""

        prompt = """
        生成 {num_cases} 个独特且多样化的想法，用于测试完成此任务的提示词：
        
        <task_description>
        {task_description}
        </task_description>

        提示词将接收以下输入：
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>
        
        每个想法应代表一个独特的场景或示例，用于测试任务的不同方面。
        
        输出格式：
        以结构化的 JSON 数组形式提供你的响应，其中每个元素是对该想法的简要描述。
        
        示例：
        ```json
        [
            "使用技术性计算机科学术语进行测试",
            "使用医学研究发现进行测试",
            "使用复杂数学概念进行测试",
            ...
        ]
        ```
        
        确保每个想法：
        - 与其他想法明显不同
        - 与任务描述相关
        - 足够具体，能够指导生成完整的测试用例
        - 能够快速解决，无需大量计算或多步处理
        - 输出不超过 400 个 token 即可解决

        请记住，只生成 {num_cases} 个独特的想法
        """

        system_prompt = "你是一名测试场景设计师，专门负责创建多样化、独特的测试场景。"

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )
        print(f"""generate_unique_ideas prompt: {rendered_prompt}""")

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=1.0,
        )

        result = json.loads(text)
        print(f"""generate_unique_ideas result: {result}""")
        return result

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """基于任务描述和特定想法生成单个测试用例"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        基于以下信息生成一个详细的提示词评估测试用例：
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        输出格式：
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["标准 1", "标准 2", ...] // 用于评估解决方案的简洁标准列表，1 到 4 项
        }}
        ```
        
        重要要求：
        - 你必须在 prompt_inputs 中仅使用这些确切的输入关键词：{allowed_keys}        
        - 不要在 prompt_inputs 中添加任何额外的关键词
        - 必须包含 allowed_input_keys 中列出的所有关键词
        - 使测试用例现实且实用
        - 包含可衡量、简洁的解决方案标准
        - 解决方案标准应仅针对任务描述和生成的 prompt_inputs 的直接要求
        - 避免过度指定超出核心任务范围的标准
        - 保持解决方案标准简单、聚焦，并直接与基本任务相关
        - 测试用例应针对提供的特定想法量身定制
        - 能够快速解决，无需大量计算或多步处理
        - 输出不超过 400 个 token 即可解决
        - 不要包含输出格式中指定字段之外的任何字段

        以下是一个示例输入和理想输出的例子：
        <sample_input>
        <sample_task_description>
        从文本段落中提取主题
        </sample_task_description>
        <sample_specific_idea>
        使用包含多个嵌套主题和子主题的文本进行测试（例如，一段关于可再生能源的文本，同时涵盖太阳能经济学、风力涡轮机技术和政策影响）
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {
            "prompt_inputs": {
                "content": "向可再生能源的转型涉及众多相互依存的维度。太阳能光伏技术经历了显著的成本降低，自 2010 年以来面板效率提高了 24%，而制造成本下降了 89%，使其在许多市场中与化石燃料具有经济竞争力。同时，风能通过创新的涡轮设计不断发展，采用碳纤维复合材料叶片和先进控制系统，在低风条件下将能量捕获提高了 35%。"
            },
            "solution_criteria": [
                "包含所有提到的主题"   
            ]
        }
        ```
        </ideal_output>
        这是理想输出，因为解决方案标准简洁，不要求超出任务描述范围的内容。
        """

        system_prompt = "你是一名测试用例创建者，专门负责设计评估场景。"

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )
        print(f"""generate_test_case rendered_prompt: {rendered_prompt}""")
        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=0.7,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea
        print(f"""generate_test_case result: {test_case}""")
        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        """基于任务描述生成测试数据集并保存到文件"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"已生成 {completed}/{total} 个测试用例")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"生成测试用例时出错: {e}")

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(dataset, f, indent=2, ensure_ascii=False)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """使用模型对测试用例的输出进行评分"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            强制性要求 - 任何违反都将导致自动失败（分数为 3 或更低）：
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        你的任务是极其严格地评估以下 AI 生成的解决方案。

        原始任务描述：
        <task_description>
        {task_description}
        </task_description>

        原始任务输入：
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        待评估的解决方案：
        <solution>
        {output}
        </solution>

        用于评估解决方案的标准：
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        评分指南：
        * 分数 1-3：解决方案未能满足一个或多个强制性要求
        * 分数 4-6：解决方案满足所有强制性要求，但在次要标准方面存在重大缺陷
        * 分数 7-8：解决方案满足所有强制性要求和大多数次要标准，但存在小问题
        * 分数 9-10：解决方案满足所有强制性和次要标准

        重要评分说明：
        * 仅根据列出的标准对输出进行评分。不要添加你自己的额外要求。
        * 如果解决方案满足所有强制性和次要标准，给予 10 分
        * 不要抱怨解决方案"仅仅"满足强制性和次要标准。解决方案不应该超出要求 - 它们应该满足确切列出的标准。
        * 任何违反强制性要求的情况都必须导致 3 分或更低的分数
        * 应充分利用 1-10 的评分范围 - 在适当时不要犹豫给予低分

        输出格式
        以结构化的 JSON 对象形式提供你的评估，包含以下字段，按此特定顺序：
        - "strengths": 1-3 个关键优势的数组
        - "weaknesses": 1-3 个需要改进的关键领域的数组
        - "reasoning": 对你整体评估的简洁解释
        - "score": 1-10 之间的数字

        以 JSON 格式响应。保持你的响应简洁直接。
        示例响应格式：
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )
        print(f"""模型评估 eval_prompt: {eval_prompt}""")

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(
            messages,
            stop_sequences=["```"],
            temperature=0.0,
        )
        result = json.loads(eval_text)
        print(f"""模型评估 result: {result}""")
        return result

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """运行测试用例并对结果进行评分"""
        prompt_result = run_prompt_function(test_case["prompt_inputs"])
        
        # 支持两种返回格式：
        # 1. (prompt, output) 元组 - 新格式，包含完整 prompt
        # 2. output 字符串 - 旧格式，仅返回输出
        if isinstance(prompt_result, tuple):
            prompt, output = prompt_result
        else:
            prompt = None
            output = prompt_result

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        result = {
            "output": output,
            "prompt": prompt,  # 保存完整的 prompt
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }
        print(f"""run_test_case result: {result}""")
        return result

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """对数据集中的所有测试用例运行评估"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"已评分 {completed}/{total} 个测试用例")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"平均分数: {average_score}")

        with open(json_output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        html = generate_prompt_evaluation_report(results, extra_criteria)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

In [113]:
# 创建 PromptEvaluator 实例
# 增加 `max_concurrent_tasks` 可以提高并发性，但要注意速率限制错误！
evaluator = PromptEvaluator(max_concurrent_tasks=1)

# 生成测试数据集

使用 `evaluator.generate_dataset()` 方法生成测试数据集。

**参数说明**：
- `task_description`: 描述你要测试的提示词的目的或目标
- `prompt_inputs_spec`: 描述提示词所需的不同输入（字典格式，键为输入名称，值为输入描述）
- `output_file`: 生成的数据集保存的文件路径（默认 "dataset.json"）
  - **文件位置**：保存在当前工作目录，完整路径：`/Users/canglong/Documents/deep-learning/building_with_claudeCode_api/dataset.json`
- `num_cases`: 要生成的测试用例数量（如果遇到速率限制错误，建议保持较低值）

**输出**：
- 运行成功后会在当前工作目录生成 `dataset.json` 文件
- 文件包含所有生成的测试用例，每个用例包含 `prompt_inputs` 和 `solution_criteria`

**示例**：本例为运动员生成一日膳食计划的提示词创建测试数据集。

In [114]:
dataset = evaluator.generate_dataset(
    # 目标
    task_description="为一名运动员编写一份紧凑、简洁的一日膳食计划",
    # 所需的不同输入
    prompt_inputs_spec={
        "height": "运动员的身高（单位：厘米）",
        "weight": "运动员的体重（单位：千克）",
        "goal": "运动员的目标",
        "restrictions": "运动员的饮食限制",
    },
    # 生成的数据集写入位置
    output_file="dataset.json",
    # 要生成的测试用例数量（如果遇到速率限制错误，建议保持较低值）
    num_cases=1,
)

generate_unique_ideas prompt: 
生成 1 个独特且多样化的想法，用于测试完成此任务的提示词：

<task_description>
为一名运动员编写一份紧凑、简洁的一日膳食计划
</task_description>

提示词将接收以下输入：
<prompt_inputs>
{prompt_inputs_spec}
</prompt_inputs>

每个想法应代表一个独特的场景或示例，用于测试任务的不同方面。

输出格式：
以结构化的 JSON 数组形式提供你的响应，其中每个元素是对该想法的简要描述。

示例：
```json
[
    "使用技术性计算机科学术语进行测试",
    "使用医学研究发现进行测试",
    "使用复杂数学概念进行测试",
    ...
]
```

确保每个想法：
- 与其他想法明显不同
- 与任务描述相关
- 足够具体，能够指导生成完整的测试用例
- 能够快速解决，无需大量计算或多步处理
- 输出不超过 400 个 token 即可解决

请记住，只生成 1 个独特的想法

chat text: 
[
    "为一名在极端寒冷环境下（如高海拔极地探险）且具有严格生酮饮食限制的专业登山运动员，设计一份包含确切宏量营养素比例和电解质补充说明的一日高热量密度膳食计划。"
]
```
generate_unique_ideas result: ['为一名在极端寒冷环境下（如高海拔极地探险）且具有严格生酮饮食限制的专业登山运动员，设计一份包含确切宏量营养素比例和电解质补充说明的一日高热量密度膳食计划。']
generate_test_case rendered_prompt: 
基于以下信息生成一个详细的提示词评估测试用例：

<task_description>
为一名运动员编写一份紧凑、简洁的一日膳食计划
</task_description>

<specific_idea>
为一名在极端寒冷环境下（如高海拔极地探险）且具有严格生酮饮食限制的专业登山运动员，设计一份包含确切宏量营养素比例和电解质补充说明的一日高热量密度膳食计划。
</specific_idea>

<allowed_input_keys>
"height", "weight", "goal", "restrictions"
</

# 定义要评估的提示词函数

`run_prompt()` 函数定义了你要评估的提示词逻辑。

**功能**：
- 接收测试用例的 `prompt_inputs` 作为输入
- 构建完整的提示词（包含任务描述、输入参数、指导原则和示例）
- 调用 `chat()` 函数获取模型输出
- 返回原始模型输出

**注意**：此函数会为每个测试用例执行一次，用于生成实际的提示词输出。

### 版本0：这个人应该吃什么？

In [115]:
# 定义并运行你要评估的提示词，返回原始模型输出
# 此函数为每个测试用例执行一次
# 返回格式: (prompt, output) 元组，其中 prompt 是最终拼接的提示词，output 是模型输出
def run_prompt0(prompt_inputs):
    prompt = f"""
    这个人应该吃什么？

    - 身高: {prompt_inputs["height"]} 
    - 体重: {prompt_inputs["weight"]} 
    - 目标: {prompt_inputs["goal"]} 
    - 饮食限制: {prompt_inputs["restrictions"]} 

    """
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, max_tokens=4000)
    print(f"""run_prompt text: {text}""")
    return (prompt, text)

### 技巧1：清晰和直接
为一名运动员生成一份符合其饮食限制的一日膳食计划。

In [116]:
# 定义并运行你要评估的提示词，返回原始模型输出
# 此函数为每个测试用例执行一次
# 返回格式: (prompt, output) 元组，其中 prompt 是最终拼接的提示词，output 是模型输出
def run_prompt1(prompt_inputs):
    prompt = f"""
    为一名运动员生成一份符合其饮食限制的一日膳食计划。

    - 身高: {prompt_inputs["height"]} 
    - 体重: {prompt_inputs["weight"]} 
    - 目标: {prompt_inputs["goal"]} 
    - 饮食限制: {prompt_inputs["restrictions"]} 

    """
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, max_tokens=4000)
    print(f"""run_prompt text: {text}""")
    return (prompt, text)

In [117]:
# 定义并运行你要评估的提示词，返回原始模型输出
# 此函数为每个测试用例执行一次
# 返回格式: (prompt, output) 元组，其中 prompt 是最终拼接的提示词，output 是模型输出
def run_prompt2(prompt_inputs):
    prompt = f"""
    为一名运动员生成一份符合其饮食限制的一日膳食计划。

    <athlete_information> 
    - 身高: {prompt_inputs["height"]} 
    - 体重: {prompt_inputs["weight"]} 
    - 目标: {prompt_inputs["goal"]} 
    - 饮食限制: {prompt_inputs["restrictions"]} 
    </athlete_information>

    指导原则：
    1. 包含准确的每日卡路里数量
    2. 显示蛋白质、脂肪和碳水化合物含量
    3. 指定每餐的用餐时间
    4. 仅使用符合限制的食物
    5. 以克为单位列出所有食物份量
    6. 如果提到预算，保持经济实惠

    以下是一个示例输入和理想输出的例子：
    <sample_input>
    身高: 170
    体重: 70
    目标: 保持健康并改善胆固醇水平
    限制: 高胆固醇
    </sample_input>
    <ideal_output>
    以下是为一名旨在保持健康并改善胆固醇水平的运动员制定的一日膳食计划：

    *   **卡路里目标：** 约 2500 卡路里
    *   **宏量营养素分解：** 蛋白质 (140g)，脂肪 (70g)，碳水化合物 (340g)

    **膳食计划：**

    *   **早餐 (7:00 AM)：** 燕麦片（80g 干重）配浆果（100g）和核桃（15g）。脱脂牛奶（240g）。
        *   蛋白质: 15g，脂肪: 15g，碳水化合物: 60g
    *   **上午加餐 (10:00 AM)：** 苹果（150g）配杏仁酱（30g）。
        *   蛋白质: 7g，脂肪: 18g，碳水化合物: 25g
    *   **午餐 (1:00 PM)：** 烤鸡胸肉（120g）沙拉配混合蔬菜（150g）、黄瓜（50g）、番茄（50g）和清淡的油醋汁（30g）。全麦面包（60g）。
        *   蛋白质: 40g，脂肪: 15g，碳水化合物: 70g
    *   **下午加餐 (4:00 PM)：** 希腊酸奶（170g，脱脂）配香蕉（120g）。
        *   蛋白质: 20g，脂肪: 0g，碳水化合物: 40g
    *   **晚餐 (7:00 PM)：** 烤三文鱼（140g）配蒸西兰花（200g）和藜麦（75g 干重）。
        *   蛋白质: 40g，脂肪: 20g，碳水化合物: 80g
    *   **晚间加餐 (9:00 PM)：** 一小把杏仁（20g）。
        *   蛋白质: 8g，脂肪: 12g，碳水化合物: 15g

    此膳食计划优先选择瘦肉蛋白来源、全谷物、水果和蔬菜，同时限制饱和脂肪和反式脂肪，以支持健康的胆固醇水平。
    </ideal_output>
    此示例膳食计划结构良好，提供了食物选择和数量的详细信息，并与运动员的目标和限制保持一致。
    """


    messages = []
    add_user_message(messages, prompt)
    # 使用更大的 max_tokens 值，确保完整的膳食计划不会被截断
    # 对于详细的膳食计划（包含多餐、营养成分等），使用 8000 tokens
    text = chat(messages, max_tokens=4000)
    print(f"""run_prompt text: {text}""")
    return (prompt, text)

In [118]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt2,
    dataset_file="dataset.json",
    extra_criteria="""
    输出应包含：
    - 每日卡路里总量
    - 宏量营养素分解
    - 包含确切食物、份量和时间的餐次
    """,
)

chat text: 以下是为一名在极地环境（-30°C）下进行高强度探险的运动员制定的生酮膳食计划。该计划专注于**极高的热量密度**、**防冻便携性**以及**深度入酮状态下的生热作用**。

### 运动员膳食计划总结
*   **卡路里目标：** 约 5100 卡路里
*   **宏量营养素比例：** 脂肪 75% (425g)，蛋白质 20% (255g)，碳水化合物 5% (约 63g)
*   **电解质补充方案：** 
    *   **钠：** 6-7g（通过加盐食物和电解质泡腾片）
    *   **钾：** 3.5g（通过牛油果粉、坚果和氯化钾补充剂）
    *   **镁：** 500mg（通过傍晚补充剂，预防极寒环境下的肌肉抽搐）

---

### 每日膳食计划

#### **1. 早餐 (6:00 AM) - 营地热能启动**
此餐旨在提供即时热量并提升体温。
*   **防弹浓缩咖啡：** 混合草饲黄油 (40g)、MCT油 (20g) 和浓缩咖啡。
*   **脱水全蛋粉配培根：** 全蛋粉 (80g) 还原后与肥培根碎 (60g) 混合煎炒，加入海盐 (2g)。
    *   *宏量：蛋白质 55g，脂肪 105g，碳水化合物 4g*

#### **2. 上午徒步燃料 (9:30 AM) - 持续供能 (便携)**
分次摄入，防止因血糖波动导致体温下降。
*   **夏威夷果：** (100g) - 极高脂肪比，且在零下温度下易于咀嚼。
*   **高脂肪牛肉干 (Biltong)：** (50g)。
    *   *宏量：蛋白质 35g，脂肪 82g，碳水化合物 8g*

#### **3. 午餐 (12:30 PM) - 高效能补给**
无需烹饪，开袋即食，高热量密度。
*   **自制生酮干酪肉饼 (Pemmican)：** 由干牛肉末 (60g) 和牛油/牛脂 (90g) 混合压制而成。
*   **硬质切达干酪 (Hard Cheddar)：** (50g) - 即使冻硬也含有极高能量。
    *   *宏量：蛋白质 58g，脂肪 110g，碳水化合物 2g*

#### **4. 下午徒步燃料 (3:30 PM) - 寒冷耐受支持**
*   **碧根果 (Pecan)：** (60g)。
*   **